# Intro

**Seminar by @Elad_Benjo**

In this notebook I will attempt to develop a new model for nowcsating of the Israeli GDP.

Our efforts would be towards a DFM, since we have various frequencies and high demensional data.

In [2]:
from google.colab import drive
import sys

drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/gdp_nowcasting_seminar/src')

Mounted at /content/drive


In [3]:
import pandas as pd

## Papers

https://www.federalreserve.gov/econres/ifdp/files/ifdp1385.pdf

https://research-api.cbs.dk/ws/portalfiles/portal/108037712/1790388_Masters_thesis_final.pdf

https://unit8co.github.io/darts/generated_api/darts.dataprocessing.transformers.midas.html

https://www.bankofengland.co.uk/macro-technical-paper/2025/nowcasting-gdp-at-the-bank-of-england-a-staggered-combination-midas-approach

https://www.bundesbank.de/resource/blob/703478/e103f573e8ad3241e7cd65431dcbaf6c/mL/2009-03-20-dkp-07-data.pdf

# OLS

In [6]:
factor_df = pd.read_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/dfm/factor_df.pkl')

In [7]:
factor_quarterly = factor_df.rolling(30).mean().resample("QE").last()

In [8]:
from statsmodels.api import OLS, add_constant


X = add_constant(factor_quarterly)
y = pd.read_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/dfm/y_train.pkl')


In [9]:
y = y['GDP']

In [10]:
X = X.loc[y.index]

In [11]:
model = OLS(y, X, missing="drop").fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                    GDP   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.009
Method:                 Least Squares   F-statistic:                   0.02022
Date:                Sun, 31 Aug 2025   Prob (F-statistic):              0.887
Time:                        23:45:17   Log-Likelihood:                 293.91
No. Observations:                 107   AIC:                            -583.8
Df Residuals:                     105   BIC:                            -578.5
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0098      0.002      6.463      0.0

# Midas

In [16]:
pip install git+https://github.com/MIDASverse/MIDASpy.git

  Cloning https://github.com/MIDASverse/MIDASpy.git to /tmp/pip-req-build-3hemu_kl
  Running command git clone --filter=blob:none --quiet https://github.com/MIDASverse/MIDASpy.git /tmp/pip-req-build-3hemu_kl
  Resolved https://github.com/MIDASverse/MIDASpy.git to commit b43f3ccedc1a150cc74c6a82d049239fe27d10ee
  Preparing metadata (setup.py) ... done
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
INFO: pip is looking at multiple versions of midaspy to determine which version is compatible with other requirements. This could take a while.
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11
ERROR: Could not find a version that satisfies the requirement tensorflow_addons<0.20 (from midaspy) (from versions: none)
ERROR: No matching

In [13]:
gdp_series = pd.read_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/dfm/y_train.pkl')['GDP']

In [14]:
factor_series = pd.read_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/dfm/factor_df.pkl')

In [15]:
from MIDASpy.regression import MIDASRegression
from MIDASpy.data import prepare_midas_data

lags = 90

# data prep
X, y = prepare_midas_data(
    y=gdp_series,                 # quarterly target
    X_list=[factor_series],       # daily factor
    X_lags=[lags],
    method="normalized_beta"
)

# initialize
midas = MIDASRegression()
midas.fit(X, y)


ModuleNotFoundError: No module named 'MIDASpy'